In [1]:
import pandas as pd
from pathlib import Path

# Load the original dataset without modifying it
file_path = Path("C:/Users/91931/Desktop/python_learning/Decoding Netflix End-to-End Content Strategy & BI Analysis/data") / "netflix_titles.csv"
df = pd.read_csv(file_path)

# Audit missing values in each column
missing_audit = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (
        df.isna().sum() / len(df) * 100
    ).round(2)
})

missing_audit = missing_audit.sort_values(
    "missing_count", ascending=False
)

print("DATASET SHAPE:", df.shape)
print("\nMISSING-VALUE AUDIT:")
print(missing_audit.to_string())

print("\nTOTAL MISSING CELLS:", int(df.isna().sum().sum()))
print("TOTAL CELLS:", df.size)
print(
    "OVERALL MISSING RATE:",
    round(df.isna().sum().sum() / df.size * 100, 2),
    "%"
)


DATASET SHAPE: (8807, 12)

MISSING-VALUE AUDIT:
              missing_count  missing_percentage
director               2634               29.91
country                 831                9.44
cast                    825                9.37
date_added               10                0.11
rating                    4                0.05
duration                  3                0.03
show_id                   0                0.00
type                      0                0.00
title                     0                0.00
release_year              0                0.00
listed_in                 0                0.00
description               0                0.00

TOTAL MISSING CELLS: 4307
TOTAL CELLS: 105684
OVERALL MISSING RATE: 4.08 %


In [2]:

# Task 2.2: Detect empty or whitespace-only strings

string_columns = df.select_dtypes(
    include=["object", "string"]
).columns

blank_audit = {}

for col in string_columns:
    blank_count = (
        df[col]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
        .sum()
    )

    if blank_count > 0:
        blank_audit[col] = int(blank_count)

print("\nEMPTY/WHITESPACE-ONLY STRING AUDIT:")

if blank_audit:
    for col, count in blank_audit.items():
        print(f"{col}: {count}")
else:
    print("No empty or whitespace-only strings found.")



EMPTY/WHITESPACE-ONLY STRING AUDIT:
No empty or whitespace-only strings found.


In [3]:

# Task 2.3: Duplicate audit

exact_duplicates = int(df.duplicated().sum())
duplicate_ids = int(df["show_id"].duplicated().sum())

print("\nDUPLICATE AUDIT:")
print("Exact duplicate rows:", exact_duplicates)
print("Duplicate show_id values:", duplicate_ids)

# Check for titles with repeated names
duplicate_titles = int(
    df["title"].duplicated(keep=False).sum()
)

print(
    "Rows with a title appearing more than once:",
    duplicate_titles
)



DUPLICATE AUDIT:
Exact duplicate rows: 0
Duplicate show_id values: 0
Rows with a title appearing more than once: 0


In [4]:

# Task 2.4: Inspect column data types and sample values

print("\nCOLUMN DATA TYPES:")
print(df.dtypes.to_string())

print("\nSAMPLE VALUES:")
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].dropna().head(3).tolist())

print("\nDATE-ADDED SAMPLE:")
print(df["date_added"].dropna().head(10).to_list())

print("\nRELEASE YEAR RANGE:")
print(df["release_year"].min(), "to", df["release_year"].max())




COLUMN DATA TYPES:
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object

SAMPLE VALUES:

show_id:
['s1', 's2', 's3']

type:
['Movie', 'TV Show', 'TV Show']

title:
['Dick Johnson Is Dead', 'Blood & Water', 'Ganglands']

director:
['Kirsten Johnson', 'Julien Leclercq', 'Mike Flanagan']

cast:
['Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng', 'Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera', 'Mayur More, Jitendra Ku

In [5]:

# Task 2.5: Validate date_added without changing the original

parsed_dates = pd.to_datetime(
    df["date_added"],
    format="%B %d, %Y",
    errors="coerce"
)

original_dates_present = df["date_added"].notna()
valid_dates = parsed_dates.notna()

invalid_dates = (
    original_dates_present & ~valid_dates
)

print("\nDATE VALIDATION AUDIT:")
print("Total records:", len(df))
print("Originally missing dates:", int((~original_dates_present).sum()))
print(
    "Valid dates:",
    int((original_dates_present & valid_dates).sum())
)
print("Invalid non-missing dates:", int(invalid_dates.sum()))

if invalid_dates.any():
    print("\nInvalid date examples:")
    print(df.loc[invalid_dates, "date_added"].head(10).to_list())

if valid_dates.any():
    print(
        "Earliest valid date:",
        parsed_dates.min().date()
    )
    print(
        "Latest valid date:",
        parsed_dates.max().date()
    )



DATE VALIDATION AUDIT:
Total records: 8807
Originally missing dates: 10
Valid dates: 8709
Invalid non-missing dates: 88

Invalid date examples:
[' August 4, 2017', ' December 23, 2018', ' December 15, 2018', ' July 1, 2017', ' July 26, 2019', ' May 26, 2016', ' November 1, 2019', ' December 2, 2017', ' March 15, 2019', ' October 1, 2019']
Earliest valid date: 2008-01-01
Latest valid date: 2021-09-25


In [6]:

# Task 2.5.1: Test date parsing after trimming whitespace
# This creates a temporary Series; it does not modify df.

trimmed_dates = df["date_added"].str.strip()

parsed_trimmed_dates = pd.to_datetime(
    trimmed_dates,
    format="%B %d, %Y",
    errors="coerce"
)

originally_present = df["date_added"].notna()
valid_after_trim = parsed_trimmed_dates.notna()

invalid_after_trim = originally_present & ~valid_after_trim

leading_or_trailing_spaces = (
    originally_present
    & df["date_added"].ne(trimmed_dates)
)

print("\nDATE VALIDATION AFTER WHITESPACE TEST:")
print(
    "Dates with leading/trailing spaces:",
    int(leading_or_trailing_spaces.sum())
)
print(
    "Valid dates after trimming:",
    int((originally_present & valid_after_trim).sum())
)
print(
    "Invalid non-missing dates after trimming:",
    int(invalid_after_trim.sum())
)

if invalid_after_trim.any():
    print(
        "Remaining invalid examples:",
        df.loc[invalid_after_trim, "date_added"].head(10).to_list()
    )



DATE VALIDATION AFTER WHITESPACE TEST:
Dates with leading/trailing spaces: 88
Valid dates after trimming: 8797
Invalid non-missing dates after trimming: 0


In [7]:

# Task 2.6: Inspect duration formats without modifying data

print("\nDURATION AUDIT:")

print("Missing duration values:", int(df["duration"].isna().sum()))

print("\nMost common duration values:")
print(df["duration"].value_counts(dropna=False).head(20).to_string())

print("\nDuration by content type:")
for content_type, group in df.groupby("type"):
    print(f"\n{content_type}:")
    print("Records:", len(group))
    print("Missing durations:", int(group["duration"].isna().sum()))
    print("Sample values:", group["duration"].dropna().head(10).to_list())



DURATION AUDIT:
Missing duration values: 3

Most common duration values:
duration
1 Season     1793
2 Seasons     425
3 Seasons     199
90 min        152
94 min        146
97 min        146
93 min        146
91 min        144
95 min        137
96 min        130
92 min        129
102 min       122
98 min        120
99 min        118
88 min        116
101 min       116
103 min       114
106 min       111
100 min       108
89 min        106

Duration by content type:

Movie:
Records: 6131
Missing durations: 3
Sample values: ['90 min', '91 min', '125 min', '104 min', '127 min', '91 min', '67 min', '94 min', '161 min', '61 min']

TV Show:
Records: 2676
Missing durations: 0
Sample values: ['2 Seasons', '1 Season', '1 Season', '2 Seasons', '1 Season', '9 Seasons', '1 Season', '1 Season', '1 Season', '4 Seasons']


In [8]:

# Task 2.7: Validate duration formats by content type

duration_text = df["duration"].astype("string").str.strip()

movie_pattern = duration_text.str.fullmatch(r"\d+\s+min").fillna(False)

tv_pattern = duration_text.str.fullmatch(
    r"\d+\s+Seasons?"
).fillna(False)

movie_rows = df["type"].eq("Movie") & df["duration"].notna()
tv_rows = df["type"].eq("TV Show") & df["duration"].notna()

invalid_movie = movie_rows & ~movie_pattern
invalid_tv = tv_rows & ~tv_pattern

print("\nDURATION FORMAT VALIDATION:")
print("Non-missing movie durations:", int(movie_rows.sum()))
print("Valid movie duration formats:", int((movie_rows & movie_pattern).sum()))
print("Invalid movie duration formats:", int(invalid_movie.sum()))

print("\nNon-missing TV show durations:", int(tv_rows.sum()))
print("Valid TV show duration formats:", int((tv_rows & tv_pattern).sum()))
print("Invalid TV show duration formats:", int(invalid_tv.sum()))

if invalid_movie.any():
    print("Invalid movie examples:")
    print(df.loc[invalid_movie, ["title", "duration"]].head(10).to_string(index=False))

if invalid_tv.any():
    print("Invalid TV show examples:")
    print(df.loc[invalid_tv, ["title", "duration"]].head(10).to_string(index=False))



DURATION FORMAT VALIDATION:
Non-missing movie durations: 6128
Valid movie duration formats: 6128
Invalid movie duration formats: 0

Non-missing TV show durations: 2676
Valid TV show duration formats: 2676
Invalid TV show duration formats: 0


In [9]:

# Task 2.8: Inspect content ratings

print("\nRATING AUDIT:")
print("Missing ratings:", int(df["rating"].isna().sum()))
print("Distinct non-missing ratings:", df["rating"].nunique())

print("\nRating frequencies:")
print(df["rating"].value_counts(dropna=False).to_string())

# Detect values that contain leading/trailing whitespace
rating_text = df["rating"].astype("string")

has_extra_spaces = (
    rating_text.notna()
    & rating_text.ne(rating_text.str.strip())
)

print(
    "\nRatings with leading/trailing spaces:",
    int(has_extra_spaces.sum())
)

# Inspect rating labels without changing the source data
print("\nDistinct rating labels:")
print(sorted(df["rating"].dropna().unique().tolist()))



RATING AUDIT:
Missing ratings: 4
Distinct non-missing ratings: 17

Rating frequencies:
rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NaN            4
NC-17          3
UR             3
66 min         1
74 min         1
84 min         1

Ratings with leading/trailing spaces: 0

Distinct rating labels:
['66 min', '74 min', '84 min', 'G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR']


In [10]:

# Task 2.8.1: Inspect suspicious rating records

suspicious_ratings = ["66 min", "74 min", "84 min"]

suspect_rows = df.loc[
    df["rating"].isin(suspicious_ratings),
    ["show_id", "title", "type", "rating", "duration"]
]

print("\nSUSPICIOUS RATING RECORDS:")
print(suspect_rows.to_string(index=False))

print("\nMISSING DURATION RECORDS:")
print(
    df.loc[
        df["duration"].isna(),
        ["show_id", "title", "type", "rating", "duration"]
    ].to_string(index=False)
)



SUSPICIOUS RATING RECORDS:
show_id                                title  type rating duration
  s5542                      Louis C.K. 2017 Movie 74 min      NaN
  s5795                Louis C.K.: Hilarious Movie 84 min      NaN
  s5814 Louis C.K.: Live at the Comedy Store Movie 66 min      NaN

MISSING DURATION RECORDS:
show_id                                title  type rating duration
  s5542                      Louis C.K. 2017 Movie 74 min      NaN
  s5795                Louis C.K.: Hilarious Movie 84 min      NaN
  s5814 Louis C.K.: Live at the Comedy Store Movie 66 min      NaN


In [11]:

# Task 2.8.2: Summarize rating anomalies by content type

suspicious_ratings = ["66 min", "74 min", "84 min"]

print("\nSUSPICIOUS RATINGS BY CONTENT TYPE:")
print(
    df.loc[df["rating"].isin(suspicious_ratings)]
    .groupby(["type", "rating"])
    .size()
    .to_string()
)

print("\nRATING AUDIT SUMMARY:")
print("Total records:", len(df))
print("Missing ratings:", int(df["rating"].isna().sum()))
print(
    "Suspicious duration-like ratings:",
    int(df["rating"].isin(suspicious_ratings).sum())
)
print(
    "Records with either a missing rating or suspicious rating:",
    int(
        (
            df["rating"].isna()
            | df["rating"].isin(suspicious_ratings)
        ).sum()
    )
)



SUSPICIOUS RATINGS BY CONTENT TYPE:
type   rating
Movie  66 min    1
       74 min    1
       84 min    1

RATING AUDIT SUMMARY:
Total records: 8807
Missing ratings: 4
Suspicious duration-like ratings: 3
Records with either a missing rating or suspicious rating: 7


In [12]:

# Task 2.9: Inspect country and genre formatting

for col in ["country", "listed_in"]:
    values = df[col].astype("string")

    extra_spaces = (
        values.notna()
        & values.ne(values.str.strip())
    )

    print(f"\n{col.upper()} AUDIT:")
    print("Missing values:", int(df[col].isna().sum()))
    print("Distinct non-missing values:", df[col].nunique())
    print("Values with leading/trailing spaces:", int(extra_spaces.sum()))

    print("\nTop 10 values:")
    print(df[col].value_counts().head(10).to_string())



COUNTRY AUDIT:
Missing values: 831
Distinct non-missing values: 748
Values with leading/trailing spaces: 0

Top 10 values:
country
United States     2818
India              972
United Kingdom     419
Japan              245
South Korea        199
Canada             181
Spain              145
France             124
Mexico             110
Egypt              106

LISTED_IN AUDIT:
Missing values: 0
Distinct non-missing values: 514
Values with leading/trailing spaces: 0

Top 10 values:
listed_in
Dramas, International Movies                        362
Documentaries                                       359
Stand-Up Comedy                                     334
Comedies, Dramas, International Movies              274
Dramas, Independent Movies, International Movies    252
Kids' TV                                            220
Children & Family Movies                            215
Children & Family Movies, Comedies                  201
Documentaries, International Movies                 186


In [13]:

# Task 2.10: Validate key fields

print("\nSTRUCTURAL CONSISTENCY AUDIT:")

print("Total records:", len(df))
print("Unique show IDs:", df["show_id"].nunique())
print("Missing show IDs:", int(df["show_id"].isna().sum()))
print("Duplicate show IDs:", int(df["show_id"].duplicated().sum()))

print("\nContent type counts:")
print(df["type"].value_counts(dropna=False).to_string())

expected_types = {"Movie", "TV Show"}
actual_types = set(df["type"].dropna().unique())

print("\nUnexpected content types:", sorted(actual_types - expected_types))
print("Missing content types:", int(df["type"].isna().sum()))



STRUCTURAL CONSISTENCY AUDIT:
Total records: 8807
Unique show IDs: 8807
Missing show IDs: 0
Duplicate show IDs: 0

Content type counts:
type
Movie      6131
TV Show    2676

Unexpected content types: []
Missing content types: 0
